# 07 — Fisher–Rao Geodesics

`02/06` left you with a curved triangle: a metric $ds^2 = \sum_i dp_i^2/p_i$ that charges $1/p_i$ for moving outcome $i$, so that steps near the boundary are expensive. This notebook turns that metric into lengths of whole paths, finds the shortest ones, and measures them. The key is one change of variable, $\phi_i = \sqrt{p_i}$, which turns the curved triangle into a piece of a sphere. From there follow the closed-form distance $d_{FR}(p, q) = 2\arccos\big(\sum_i\sqrt{p_i q_i}\big)$, the finite distance to a corner, and a new reading of the Bhattacharyya overlap and the Hellinger distance. You finish by computing a geodesic by hand and measuring how far it bulges.

**Prerequisites:** `02/06_fisher_rao_metric` (the metric $ds^2 = \sum_i dp_i^2/p_i$ on tangent steps with $\sum_i dp_i = 0$, the coin's $I(\theta) = 1/\theta(1-\theta)$, and the integral $\int d\theta/\sqrt{\theta(1-\theta)}$ that converges at a corner), `02/05_the_probability_simplex` (the triangle, the mixture path $(1-t)p + tq$, and the Hellinger convention $\mathrm{Hel} = \sqrt{1 - B}$), `01/14` (the Bhattacharyya overlap $B = \sum_i\sqrt{p_i q_i}$).

**What you'll be able to do**
- Derive $ds^2 = 4\sum_i d\phi_i^2$ from the Fisher metric with $\phi_i = \sqrt{p_i}$, explain where the factor $2$ of the distance comes from, and check the result on a coin against the integral $2\arcsin\sqrt\theta$.
- Compute $d_{FR}$ by hand, state the distance $2\arccos\sqrt{p_i}$ from a distribution to the corner $\delta_i$, and say why no two distributions are more than $\pi$ apart.
- Read the Bhattacharyya coefficient as $\cos(d_{FR}/2)$ and the Hellinger distance as a chord of the sphere, and deduce the Hellinger triangle inequality from it.
- Compute a geodesic point by hand (spherical interpolation, then squaring), explain why the geodesic bulges away from an edge, and give the size of that bulge in numbers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.geometry.info_geometry import (
    fisher_rao_distance,
    fisher_rao_geodesic,
    mixture_interpolation,
)

rng = np.random.default_rng(7)   # random pairs and triples of distributions in section 2
viz.use_course_style()

V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])   # the triangle of 02/05: corners delta_1, delta_2, delta_3 in the plane

## 1. The square-root map: the curved triangle is a piece of a sphere

The Fisher metric charges $1/p_i$ for moving outcome $i$. Absorb that weight into the coordinates: the square root does it, because its slope $1/(2\sqrt p)$ is steep exactly where $p$ is small, so it stretches the region near the boundary by the amount the metric asks for.

**Length of a path.** A path $p(t)$, $t \in [0, 1]$, has the length $L = \int_0^1 \sqrt{\sum_i \dot p_i(t)^2/p_i(t)}\,dt$: the metric of `02/06` summed along it. The distance $d_{FR}(p, q)$ is the length of the shortest path from $p$ to $q$.

**The derivation, in two lines.** Put $\phi_i = \sqrt{p_i}$. Then $p_i = \phi_i^2$ and $dp_i = 2\phi_i\,d\phi_i$, so each term of the metric becomes
$$
\frac{dp_i^2}{p_i} = \frac{4\phi_i^2\,d\phi_i^2}{\phi_i^2} = 4\,d\phi_i^2, \qquad\text{hence}\qquad ds^2 = \sum_i \frac{dp_i^2}{p_i} = 4\sum_i d\phi_i^2 .
$$
The constraint $\sum_i p_i = 1$ becomes $\sum_i \phi_i^2 = 1$: the vector $\phi = \sqrt p$ lives on the **unit sphere**, in the part where every coordinate is non-negative (a quarter circle for a coin, an eighth of a sphere for three outcomes). The tangent condition $\sum_i dp_i = 0$ reads $\sum_i \phi_i\,d\phi_i = 0$, which says that $d\phi$ is tangent to the sphere, as it should be. So the Fisher length of a path in the triangle is **twice** the ordinary length of its image on the unit sphere. On a unit sphere, the shortest path between two points is the arc of the great circle through them (the circle cut by a plane through the centre), and its length is the angle between the two position vectors, $\arccos(\phi\cdot\psi)$, here with $\phi\cdot\psi = \sum_i\sqrt{p_i q_i}$. Since both vectors have non-negative entries, that angle is at most $\pi/2$, and the arc stays in the non-negative part of the sphere (section 3: both interpolation coefficients are then non-negative), so it is the image of a path in the triangle. That gives the closed form
$$
d_{FR}(p, q) = 2\arccos\Big(\sum_i \sqrt{p_i\,q_i}\Big) \in [0, \pi] .
$$
The factor $2$ is the $4$ of $ds^2 = 4\sum_i d\phi_i^2$ under a square root: in the Fisher metric the triangle is a piece of a sphere of radius $2$, not $1$.

**A check you can do on paper: the coin.** `02/06` showed that the Fisher length from the corner "always tails" ($\theta = 0$) to a coin $\theta$ is the integral $\int_0^\theta dt/\sqrt{t(1-t)}$, and that it converges. Substitute $t = \sin^2 u$: then $dt = 2\sin u\cos u\,du$ and $\sqrt{t(1-t)} = \sin u\cos u$, so the integrand becomes $2\,du$ and
$$
\int_0^\theta \frac{dt}{\sqrt{t(1-t)}} = 2\arcsin\sqrt\theta .
$$
The sphere gives it without an integral: the coin's square-root vector is $(\sqrt\theta, \sqrt{1-\theta}) = (\sin u, \cos u)$ with $u = \arcsin\sqrt\theta$, a point on the quarter circle at angle $u$ from the tails corner $(0, 1)$; twice the angle is $2\arcsin\sqrt\theta$.

**Corners are at a finite distance.** The overlap of $p$ with the corner $\delta_i$ has a single term, $\sqrt{p_i}$, so
$$
d_{FR}(\delta_i, p) = 2\arccos\sqrt{p_i} .
$$
It is finite for every $p$: the coefficient $1/p_i$ of the metric blows up at the boundary, but the length element $ds$ grows only like $1/\sqrt{p_i}$, too slowly to push the corners to infinity. Two corners are at $2\arccos 0 = \pi$, and $\pi$ is the largest distance there is, because square-root vectors have non-negative entries and so are never more than a right angle apart. The distance reaches $\pi$ exactly when $\sum_i\sqrt{p_i q_i} = 0$, that is, when the two distributions have disjoint supports. The cell below checks the integral by quadrature against $2\arcsin\sqrt\theta$ and the helper `fisher_rao_distance`, the corner distances, and the two exact lengths that `02/06` quoted from this notebook.

In [ ]:
# The coin: Fisher length from the tails corner (theta = 0) to theta, three ways
print(f"{'theta':>6s} {'quadrature of dt/sqrt(t(1-t))':>31s} {'2 arcsin sqrt(theta)':>21s} {'fisher_rao_distance':>20s}")
for theta in [0.1, 0.25, 0.5, 0.9, 1.0]:
    integral, _ = quad(lambda t: 1.0 / np.sqrt(t * (1.0 - t)), 0.0, theta)   # quad copes with the 1/sqrt blow-up at the ends
    closed = 2 * np.arcsin(np.sqrt(theta))
    helper = fisher_rao_distance([0.0, 1.0], [theta, 1.0 - theta])          # (heads, tails): the corner "always tails" is (0, 1)
    print(f"{theta:6.2f} {integral:31.6f} {closed:21.6f} {helper:20.6f}")

# Corners: d(delta_i, p) = 2 arccos sqrt(p_i)
uniform3 = np.ones(3) / 3
print(f"\ncorner delta_1 to the uniform (1/3, 1/3, 1/3): helper {fisher_rao_distance([1, 0, 0], uniform3):.4f},  2 arccos sqrt(1/3) = {2 * np.arccos(np.sqrt(1 / 3)):.4f}")
p = np.array([0.7, 0.2, 0.1])
print(f"corner delta_1 to (0.7, 0.2, 0.1):            helper {fisher_rao_distance([1, 0, 0], p):.4f},  2 arccos sqrt(0.7)  = {2 * np.arccos(np.sqrt(0.7)):.4f}")
print(f"corner delta_1 to corner delta_2:             helper {fisher_rao_distance([1, 0, 0], [0, 1, 0]):.4f}  (pi = {np.pi:.4f}, disjoint supports)")
print(f"(0.5, 0.5, 0) to (0, 0, 1):                   helper {fisher_rao_distance([0.5, 0.5, 0], [0, 0, 1]):.4f}  (disjoint supports again)")
P, Q = rng.dirichlet(np.ones(3), size=5000), rng.dirichlet(np.ones(3), size=5000)
d_random = np.array([fisher_rao_distance(a, b) for a, b in zip(P, Q)])
print(f"largest distance among 5000 random pairs of the triangle: {d_random.max():.4f}  (below pi)")

# The two moves of 0.02 from 02/06, now measured exactly
p_edge = np.array([0.5, 0.48, 0.02])
d_common = fisher_rao_distance(p_edge, p_edge + np.array([-0.02, 0.02, 0.0]))
d_rare = fisher_rao_distance(p_edge, p_edge + np.array([0.0, 0.02, -0.02]))
print(f"\n02/06's moves from (0.5, 0.48, 0.02): between the common outcomes {d_common:.4f}, emptying the rare outcome {d_rare:.4f}, ratio {d_rare / d_common:.2f}")

# Equal steps in theta are unequal arcs on the unit circle (arc = half the Fisher-Rao distance)
arc = lambda a, b: np.arcsin(np.sqrt(b)) - np.arcsin(np.sqrt(a))
print(f"arc on the unit circle from theta = 0 to 0.1: {arc(0.0, 0.1):.4f};  from 0.4 to 0.6: {arc(0.4, 0.6):.4f}")
h = 1e-6
print(f"slope of 2 arcsin sqrt(theta) at theta = 1/2: {(2 * np.arcsin(np.sqrt(0.5 + h)) - 2 * np.arcsin(np.sqrt(0.5 - h))) / (2 * h):.3f}  (= sqrt(I(1/2)) = 2)")

# Figure: the coin on its quarter circle, and the distance from the tails corner
thetas_marked = np.linspace(0.0, 1.0, 11)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"wspace": 0.3})
ax = axes[0]
u = np.linspace(0, np.pi / 2, 200)
ax.plot(np.sin(u), np.cos(u), color=COLORS["muted"], lw=1.5, label="the quarter circle  phi_H^2 + phi_T^2 = 1")
ax.scatter(np.sqrt(thetas_marked), np.sqrt(1 - thetas_marked), color=viz.SOURCE_COLOR, s=46, zorder=3,
           label="theta = 0, 0.1, ..., 1  (equal steps in theta)")
for th in [0.0, 0.1, 0.5, 0.9, 1.0]:
    ax.annotate(f"theta = {th:g}", (np.sqrt(th), np.sqrt(1 - th)), xytext=(8, 4), textcoords="offset points", fontsize=9)
ax.set_aspect("equal"); ax.set_xlim(-0.05, 1.35); ax.set_ylim(-0.05, 1.2)
ax.set_xlabel("phi_H = sqrt(theta)"); ax.set_ylabel("phi_T = sqrt(1 - theta)")
ax.set_title("A coin is a point on a quarter circle", pad=10); ax.legend(loc="upper right", fontsize=9)
ax = axes[1]
grid = np.linspace(0.0, 1.0, 400)
ax.plot(grid, 2 * np.arcsin(np.sqrt(grid)), color=viz.SOURCE_COLOR, lw=2.4, label="2 arcsin sqrt(theta)  (the sphere)")
quad_pts = [quad(lambda t: 1.0 / np.sqrt(t * (1.0 - t)), 0.0, th)[0] for th in thetas_marked]
ax.scatter(thetas_marked, quad_pts, color=viz.TARGET_COLOR, s=40, zorder=3, label="quadrature of the 02/06 integral")
ax.axhline(np.pi, color=COLORS["muted"], ls=":", lw=1.2, label="pi: the other corner, the largest distance")
ax.set_xlabel("coin parameter theta"); ax.set_ylabel("Fisher-Rao distance from theta = 0")
ax.set_title("Distance from the corner 'always tails'", pad=10); ax.legend(loc="lower right", fontsize=9)
plt.show()

**Read the output and the figure.** Quadrature, closed form and helper agree to six decimals. A fair coin sits at $\pi/2 = 1.5708$ from the tails corner: halfway along the quarter circle of radius $2$ that joins the two corners, whose length is $\pi$. Left: equal steps of $0.1$ in $\theta$ are **not** equal steps on the circle. The first step, from $\theta = 0$ to $0.1$, covers an arc of $0.3218$ on the unit circle (half the distance $0.6435$ printed above), while the step from $0.4$ to $0.6$ covers only $0.2014$ for twice the change in $\theta$. The square root stretches the ends: the expense near the boundary made visible. Right: the curve is steep at both ends and close to a straight line in the middle, where its slope is $2.000$ at $\theta = \tfrac12$. The steep ends are the expense near the boundary; the finite height $\pi$ says that this expense can be summed. On three outcomes the same formula gives $1.9106$ from a corner to the uniform distribution and $1.1593$ from $\delta_1$ to $(0.7, 0.2, 0.1)$; distributions with disjoint supports sit at exactly $\pi$, and five thousand random pairs never got further than $2.5973$. Last, `02/06`'s two moves of $0.02$ have exact lengths $0.0404$ and $0.2845$, a ratio of $7.04$, as `02/06` stated.

## 2. Angle and chord: the Bhattacharyya coefficient and the Hellinger distance

Two points on a sphere can be compared along the surface (the arc) or straight through (the chord). The Fisher–Rao distance is the first; the second is a distance you met in `02/05`.

**The angle.** On the unit sphere of square roots, the inner product of $\phi = \sqrt p$ and $\psi = \sqrt q$ is the Bhattacharyya coefficient of `01/14`, $B = \sum_i\sqrt{p_i q_i}$, and it equals the cosine of the angle $\alpha$ between them. Since $d_{FR} = 2\alpha$,
$$
B = \cos\alpha = \cos\frac{d_{FR}}{2} .
$$
The overlap is the cosine of **half** the distance, because the distance is the angle measured on the sphere of radius $2$.

**The chord.** The straight-line distance between the two points of the unit sphere is $\|\phi - \psi\|^2 = \|\phi\|^2 + \|\psi\|^2 - 2\,\phi\cdot\psi = 2(1 - B)$, so
$$
\|\sqrt p - \sqrt q\|_2 = \sqrt{2(1 - B)} = 2\sin\frac{\alpha}{2} = 2\sin\frac{d_{FR}}{4}, \qquad \mathrm{Hel}(p, q) = \sqrt{1 - B} = \frac{1}{\sqrt2}\,\|\sqrt p - \sqrt q\|_2 .
$$
The Hellinger distance in this course's convention is the chord divided by $\sqrt2$. This is why it obeys the triangle inequality, as `02/05` announced: it is a Euclidean distance between the points $\sqrt p$, rescaled, and Euclidean distances obey it. The Fisher–Rao distance obeys it for a reason of the same kind: angles between unit vectors obey the triangle inequality, and $d_{FR}$ is twice an angle. A chord is never longer than its arc, and the two agree for nearby points, where $2\sin(d/4) \approx d/2$. Squaring gives $\mathrm{Hel}^2 \approx d_{FR}^2/8$, the coefficient $\tfrac18$ that `02/06` found by Taylor expansion, now read off the sphere.

**By hand.** Take the uniform $u = (\tfrac13, \tfrac13, \tfrac13)$ and $q = (0.7, 0.2, 0.1)$. Then $\sqrt{1/3} = 0.57735$, and $\sqrt{0.7} + \sqrt{0.2} + \sqrt{0.1} = 0.83666 + 0.44721 + 0.31623 = 1.60010$, so $B = 0.57735 \times 1.60010 = 0.92382$. The angle is $\arccos 0.92382 = 0.39286$, and the distance is twice that, $d_{FR} = 0.78571$. The chord is $\sqrt{2 \times 0.07618} = 0.39034$ and the Hellinger distance $\sqrt{0.07618} = 0.27601$. The cell below repeats this, checks both identities on random pairs, and checks the triangle inequality for $d_{FR}$ and for $\mathrm{Hel}$ on random triples (a check of the two arguments above, not a proof).

In [ ]:
u3 = np.ones(3) / 3
q = np.array([0.7, 0.2, 0.1])
B_hand = np.sum(np.sqrt(u3 * q))                      # inner product of the two square-root vectors
d_hand = 2 * np.arccos(B_hand)                        # twice the angle
chord = np.linalg.norm(np.sqrt(u3) - np.sqrt(q))      # straight line through the unit sphere
hel = np.sqrt(1 - B_hand)                             # the Hellinger convention of 02/05
print(f"by hand: B = {B_hand:.5f}, angle = {np.arccos(B_hand):.5f}, d_FR = {d_hand:.5f}, chord = {chord:.5f}, Hel = {hel:.5f}")
print(f"helper : fisher_rao_distance(u, q) = {fisher_rao_distance(u3, q):.5f}")
print(f"identities: cos(d/2) = {np.cos(d_hand / 2):.5f} = B;   2 sin(d/4) = {2 * np.sin(d_hand / 4):.5f} = chord;   sqrt(2) Hel = {np.sqrt(2) * hel:.5f}")

# Many pairs, spread from near-identical to near-disjoint (Dirichlet(0.4) puts mass near the edges)
P, Q = rng.dirichlet(0.4 * np.ones(3), size=3000), rng.dirichlet(0.4 * np.ones(3), size=3000)
d_all = np.array([fisher_rao_distance(a, b) for a, b in zip(P, Q)])
B_all = np.sum(np.sqrt(P * Q), axis=1)
chord_all = np.linalg.norm(np.sqrt(P) - np.sqrt(Q), axis=1)
print(f"\nover 3000 random pairs: max |B - cos(d/2)| = {np.abs(B_all - np.cos(d_all / 2)).max():.1e},  max |chord - 2 sin(d/4)| = {np.abs(chord_all - 2 * np.sin(d_all / 4)).max():.1e}")
print(f"chord never exceeds the arc on the unit sphere: max (chord - d/2) = {(chord_all - d_all / 2).max():.1e}")
print(f"at d = pi (disjoint supports): arc on the unit sphere pi/2 = {np.pi / 2:.3f}, chord 2 sin(pi/4) = {2 * np.sin(np.pi / 4):.3f}, arc - chord = {np.pi / 2 - np.sqrt(2):.3f}")
print(f"at d = 1: arc - chord = {0.5 - 2 * np.sin(0.25):.5f},  d^3/192 = {1 / 192:.5f}")

# Triangle inequality on random triples, for d_FR and for Hel
R = rng.dirichlet(0.4 * np.ones(3), size=3000)
hel_f = lambda a, b: np.sqrt(max(0.0, 1.0 - np.sum(np.sqrt(a * b))))
slack_d = min(fisher_rao_distance(a, c) + fisher_rao_distance(c, b) - fisher_rao_distance(a, b) for a, b, c in zip(P, Q, R))
slack_h = min(hel_f(a, c) + hel_f(c, b) - hel_f(a, b) for a, b, c in zip(P, Q, R))
print(f"triangle inequality over 3000 triples: smallest slack d(p,r) + d(r,q) - d(p,q) = {slack_d:.2e};  for Hel = {slack_h:.2e}  (never negative)")

# Small distances: Hel^2 / d_FR^2 -> 1/8, the coefficient of 02/06
v = np.array([0.5, -0.2, -0.3])                      # a tangent direction: entries sum to 0
for h in [0.1, 0.01, 0.001]:
    qq = q + h * v
    print(f"h = {h:5.3f}: Hel^2 / d_FR^2 = {hel_f(q, qq) ** 2 / fisher_rao_distance(q, qq) ** 2:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"wspace": 0.28})
dd = np.linspace(0, np.pi, 300)
ax = axes[0]
ax.scatter(d_all, B_all, s=6, color=viz.SOURCE_COLOR, alpha=0.5, label="3000 random pairs")
ax.plot(dd, np.cos(dd / 2), color=viz.TARGET_COLOR, lw=2, label="cos(d/2)")
ax.set_xlabel("Fisher-Rao distance d"); ax.set_ylabel("Bhattacharyya coefficient B")
ax.set_title("The overlap is the cosine of half the distance", pad=10); ax.legend(loc="lower left")
ax = axes[1]
ax.scatter(d_all, d_all / 2 - chord_all, s=6, color=viz.SOURCE_COLOR, alpha=0.5, label="arc d/2 minus chord sqrt(2) Hel, random pairs")
ax.plot(dd, dd / 2 - 2 * np.sin(dd / 4), color=viz.TARGET_COLOR, lw=2, label="d/2 - 2 sin(d/4)")
ax.plot(dd, dd ** 3 / 192, color=COLORS["muted"], ls=":", lw=1.4, label="small-d approximation d^3/192")
ax.set_xlabel("Fisher-Rao distance d"); ax.set_ylabel("arc minus chord (unit sphere)")
ax.set_title("Arc minus chord", pad=10); ax.legend(loc="upper left", fontsize=9)
plt.show()

**Read the output and the figure.** The by-hand numbers match the helper: $B = 0.92382$, $d_{FR} = 0.78571$, chord $0.39034$, Hellinger $0.27601$. Across three thousand random pairs the identities $B = \cos(d/2)$ and chord $= 2\sin(d/4)$ hold to rounding error, and on random triples neither the Fisher–Rao distance nor the Hellinger distance breaks the triangle inequality, as the two arguments above guarantee (the smallest slack over three thousand triples is $5.6\times10^{-6}$ for $d_{FR}$ and $0.0024$ for $\mathrm{Hel}$, both positive), and the chord never exceeds the arc. Left: every pair falls on the single curve $\cos(d/2)$. The overlap reaches $0$ exactly at $d = \pi$, the disjoint pairs. Right: the arc minus the chord, $d/2 - 2\sin(d/4)$, for every pair. It is never negative, stays tiny for nearby points (it starts like $d^3/192$: $0.00519$ against $0.00521$ at $d = 1$), and reaches $\pi/2 - \sqrt2 = 0.157$ at $d = \pi$, where the arc is $1.571$ and the chord $1.414$. For small steps the ratio $\mathrm{Hel}^2/d_{FR}^2$ settles on $0.125$, the $\tfrac18$ of `02/06`. Hellinger and Fisher–Rao are the chord and the arc of one sphere, not two rival rulers. `02/12` meets this same chord again when the distributions become density matrices.

## 3. Geodesics: great circles, computed by hand, and how far they bulge

A shortest path on the sphere is an arc of the great circle through its endpoints, and the Fisher–Rao geodesic is the image of that arc in the triangle. To compute a point on it, interpolate on the sphere, then square to come back. The interpolation that walks a great circle at constant speed is the **spherical linear interpolation**, or slerp. With $\alpha = \arccos B$ the angle between $\phi = \sqrt p$ and $\psi = \sqrt q$,
$$
\phi_t = \frac{\sin\big((1-t)\alpha\big)}{\sin\alpha}\,\sqrt p + \frac{\sin(t\alpha)}{\sin\alpha}\,\sqrt q, \qquad p_t = \phi_t^2 \;(\text{entry by entry}), \qquad t \in [0, 1] .
$$
The formula is for two distinct points, $\alpha > 0$ (the helper returns the point itself when they coincide). When $\alpha \le \pi/2$ both coefficients are non-negative, so $\phi_t$ keeps non-negative entries. The coefficients are chosen so that $\phi_t$ stays a unit vector and its angle to $\sqrt p$ is $t\alpha$, so it covers the arc at constant speed; the squared entries then sum to $1$, and $p_t$ is a distribution. At the midpoint both coefficients equal $\sin(\alpha/2)/\sin\alpha$, so $\phi_{1/2}$ is the normalised sum $(\sqrt p + \sqrt q)/\|\sqrt p + \sqrt q\|$, with $\|\sqrt p + \sqrt q\|^2 = 2 + 2B$. That gives a midpoint you can compute on paper:
$$
(p_{1/2})_i = \frac{(\sqrt{p_i} + \sqrt{q_i})^2}{2(1 + B)} .
$$

**By hand on three outcomes.** Take $p_0 = (0.10, 0.85, 0.05)$ and $p_1 = (0.85, 0.10, 0.05)$, two points near opposite ends of the bottom edge and at the same small height $p_3 = 0.05$. Their overlap is $B = 2 \times 0.29155 + 0.05 = 0.63310$. The midpoint formula gives $(p_{1/2})_3 = (2\sqrt{0.05})^2/(2 \times 1.63310) = 0.2/3.26619 = 0.06123$ and $(p_{1/2})_1 = (p_{1/2})_2 = (0.31623 + 0.92195)^2/3.26619 = 0.46938$. The mixture midpoint is $(0.475, 0.475, 0.05)$: the geodesic has moved mass **toward** outcome $3$. For any outcome with the same probability $c$ at both ends, the formula gives $(p_{1/2})_i = 4c/(2(1 + B)) = 2c/(1 + B)$, more than $c$ whenever $B < 1$.

**Why it bulges: the flight-route picture.** On the eighth of a sphere, the bottom edge $p_3 = 0$ is an equator, the corner $\delta_3$ is the pole, and the mixture path between $p_0$ and $p_1$ keeps $\phi_3 = \sqrt{0.05}$ fixed: it runs along a parallel of latitude. A great circle between two points of the same latitude swings toward the pole, which is why a flight between two cities at the same northern latitude flies further north than either. The metric can say it too. Write $\phi_1 = r\cos\omega$, $\phi_2 = r\sin\omega$ and $\phi_3 = \sqrt{1 - r^2}$, so that $\omega$ records how the mass $r^2 = p_1 + p_2$ is shared between outcomes $1$ and $2$. Then $d\phi_1^2 + d\phi_2^2 = dr^2 + r^2 d\omega^2$ and $d\phi_3 = -r\,dr/\sqrt{1 - r^2}$, so $dr^2 + r^2dr^2/(1 - r^2) = dr^2/(1 - r^2)$ and
$$
ds^2 = \frac{4\,dr^2}{1 - r^2} + 4(p_1 + p_2)\,d\omega^2 .
$$
The metric reads as two costs. Re-sharing between outcomes $1$ and $2$ costs $4(p_1 + p_2)\,d\omega^2$, less when less mass is shared. Moving mass to or from outcome $3$ costs $4\,dr^2/(1 - r^2)$, which diverges at $r = 1$, the bottom edge. The geodesic balances the two: it moves a little mass to outcome $3$ to make the re-sharing cheaper, which is why it bows into the interior. Between two points of one edge, leaving the edge would cost infinitely much per step, so that geodesic stays on the edge.

The cell below computes three slerp points by hand, compares them with `fisher_rao_geodesic`, and measures the bulge: the midpoint height $p_3$, the largest distance from the geodesic to the straight segment, and the Fisher length of each path, computed on the sphere as $L = 2\sum_k\|\phi_{k+1} - \phi_k\|$ over small steps. A scan over random pairs follows. A second pair, $q_0 = (0.95, 0, 0.05) \to q_1 = (0, 0.95, 0.05)$, starts and ends on the two side edges and so has a much smaller overlap.

In [ ]:
p0, p1 = np.array([0.10, 0.85, 0.05]), np.array([0.85, 0.10, 0.05])
s0, s1 = np.sqrt(p0), np.sqrt(p1)
B01 = s0 @ s1
alpha = np.arccos(B01)
print(f"B = {B01:.5f},  angle = {alpha:.5f},  d_FR = 2 x angle = {2 * alpha:.5f}  (helper {fisher_rao_distance(p0, p1):.5f})")
print(f"midpoint from the paper formula (sqrt p + sqrt q)^2 / (2(1 + B)): {np.round((s0 + s1) ** 2 / (2 * (1 + B01)), 5)}")
for t in [0.25, 0.5, 0.75]:
    phi_t = (np.sin((1 - t) * alpha) * s0 + np.sin(t * alpha) * s1) / np.sin(alpha)   # slerp on the unit sphere
    p_t = phi_t ** 2                                                                  # back to the triangle
    print(f"t = {t:4.2f}: |phi_t| = {np.linalg.norm(phi_t):.6f}   slerp by hand {np.round(p_t, 5)}   helper {np.round(fisher_rao_geodesic(p0, p1, t), 5)}   mixture {np.round(mixture_interpolation(p0, p1, t), 5)}")

def fisher_length(path):
    # the square-root map as a measuring tool: Fisher length = 2 x length on the unit sphere
    return 2 * np.sum(np.linalg.norm(np.diff(np.sqrt(path), axis=0), axis=1))

ts = np.linspace(0.0, 1.0, 4001)
pairs = {"A: (0.10, 0.85, 0.05) -> (0.85, 0.10, 0.05)": (p0, p1),
         "B: (0.95, 0, 0.05) -> (0, 0.95, 0.05)": (np.array([0.95, 0.0, 0.05]), np.array([0.0, 0.95, 0.05]))}
paths = {}
for name, (a, b) in pairs.items():
    geo = np.array([fisher_rao_geodesic(a, b, t) for t in ts])
    mix = np.array([mixture_interpolation(a, b, t) for t in ts])
    paths[name] = (geo, mix)
    Bab = np.sum(np.sqrt(a * b))
    A2, B2, G2 = a @ V, b @ V, geo @ V                               # in the plane of the triangle (side = 1)
    s = np.clip((G2 - A2) @ (B2 - A2) / np.sum((B2 - A2) ** 2), 0.0, 1.0)   # project each geodesic point on the segment, clipped to its ends
    gap = np.linalg.norm(G2 - (A2 + s[:, None] * (B2 - A2)), axis=1).max()
    Lg, Lm = fisher_length(geo), fisher_length(mix)
    print(f"\npair {name}:  B = {Bab:.4f},  d_FR = {fisher_rao_distance(a, b):.4f}")
    print(f"  midpoint p_3: geodesic {fisher_rao_geodesic(a, b, 0.5)[2]:.4f}, mixture {mixture_interpolation(a, b, 0.5)[2]:.4f}, factor 2/(1 + B) = {2 / (1 + Bab):.3f}")
    print(f"  largest distance from the geodesic to the straight segment: {gap:.4f}   (check: (p_3 mid - 0.05) x sqrt(3)/2 = {(fisher_rao_geodesic(a, b, 0.5)[2] - 0.05) * np.sqrt(3) / 2:.4f})")
    print(f"  Fisher length: geodesic {Lg:.4f}, mixture path {Lm:.4f}  (the straight path is {100 * (Lm / Lg - 1):.2f}% longer)")

# Cross-check on pair A: sum the metric itself, sqrt(sum dp^2 / p), along the geodesic
geoA = paths[list(pairs)[0]][0]
mids, dps = (geoA[1:] + geoA[:-1]) / 2, np.diff(geoA, axis=0)
print(f"\npair A, length from the metric sum_k sqrt(sum_i dp_i^2 / p_i) = {np.sum(np.sqrt(np.sum(dps ** 2 / mids, axis=1))):.5f}")
print(f"geodesic midpoint between the corners delta_1 and delta_2: {fisher_rao_geodesic([1, 0, 0], [0, 1, 0], 0.5)}")

# Excess Fisher length of the straight path over the geodesic, for random pairs of the triangle
scan_rng = np.random.default_rng(11)   # fixed seed for the scan
Ps, Qs = scan_rng.dirichlet(np.ones(3), size=2000), scan_rng.dirichlet(np.ones(3), size=2000)
tcol = ts[:, None]
excess = np.array([fisher_length((1 - tcol) * a + tcol * b) / fisher_rao_distance(a, b) - 1 for a, b in zip(Ps, Qs)]) * 100
k = excess.argmax()
print(f"\nstraight path vs geodesic over 2000 uniform random pairs: excess length median {np.median(excess):.2f}%, 95th percentile {np.percentile(excess, 95):.2f}%, maximum {excess.max():.2f}%")
print(f"  the maximum: {np.round(Ps[k], 3)} -> {np.round(Qs[k], 3)}  (smallest entries {Ps[k].min():.3f} and {Qs[k].min():.3f}, d_FR = {fisher_rao_distance(Ps[k], Qs[k]):.3f})")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.6), gridspec_kw={"wspace": 0.22, "width_ratios": [1.1, 1]})
(geoA, mixA), (geoB, mixB) = paths.values()
viz.plot_simplex_paths(
    {"mixture A, B": mixB, "geodesic A": geoA, "geodesic B": geoB},   # both mixture paths lie on the line p_3 = 0.05; drawn first so the geodesics sit on top
    endpoints={"$p_0$": p0, "$p_1$": p1, "$q_0$": pairs[list(pairs)[1]][0], "$q_1$": pairs[list(pairs)[1]][1]},
    ax=axes[0],
    title="Fisher-Rao geodesics and straight mixture paths",
)
ax = axes[1]
ax.plot(ts, geoA[:, 2], color=viz.SOURCE_COLOR, lw=2.4, label="geodesic A")
ax.plot(ts, mixA[:, 2], color=viz.FLOW_COLOR, lw=2, ls="--", label="mixture A and B (flat at 0.05)")
ax.plot(ts, geoB[:, 2], color=viz.TARGET_COLOR, lw=2.4, label="geodesic B")
for name, (a, b) in pairs.items():
    ax.scatter([0.5], [2 * a[2] / (1 + np.sum(np.sqrt(a * b)))], color=COLORS["text"], s=36, zorder=4)
ax.set_xlabel("position along the path, t"); ax.set_ylabel("probability of outcome 3")
ax.set_ylim(0.0, 0.12)
ax.set_title("The bulge: height p_3 along the path", pad=10); ax.legend(loc="lower center", fontsize=9)
plt.show()

**Read the output and the figure.** The three slerp points computed by hand coincide with the helper, and $\|\phi_t\| = 1$ at every $t$, so the squared entries are a distribution without renormalising. The paper formula reproduces the midpoint $(0.46938, 0.46938, 0.06123)$. Left: both pairs have their endpoints on the line $p_3 = 0.05$, so their two mixture paths lie on one green segment (pair A uses its middle part, pair B all of it). The geodesics (purple for A, amber for B) bow upward, away from the bottom edge, toward the pole $\delta_3$. At the scale of the triangle the bow of A is almost invisible; the right panel is the magnified view of the same rise, from $0.05$ to $0.0612$. Right: the height $p_3$ along each path. On pair A the mixture stays flat at $0.05$ while the geodesic rises to $0.0612$ at the midpoint, $2/(1 + B) = 1.225$ times the endpoint value (the black dots are the paper formula). On pair B, whose overlap is only $0.05$, the factor is $1.905$ and $p_3$ climbs to $0.0952$.

Now the size of the effect. On pair A the geodesic never gets further than $0.0097$ from the straight segment (the triangle has side $1$), which is the midpoint rise $0.0112$ times $\sqrt3/2$; the straight path is $0.19\%$ longer ($1.7738$ against $1.7705$). On pair B the largest distance is $0.0392$ and the straight path is $0.67\%$ longer ($3.0620$ against $3.0416$). The metric-sum cross-check ($1.77050$) confirms that the geodesic's length is the closed-form distance. Over two thousand random pairs, the straight path is longer by a median of $0.16\%$, by $1.66\%$ at the 95th percentile and by $4.23\%$ at most; the worst pair, $(0.530, 0.467, 0.003) \to (0.008, 0.485, 0.507)$, joins two points that each sit close to a different edge. A path close to a geodesic is only slightly longer (the excess grows like the square of the deviation, exercise 4), so these small numbers are expected. Last line: between two corners the geodesic midpoint is $(0.5, 0.5, 0)$, and the path never leaves the edge.

## Your turn

1. **Corners in higher dimension.** For $n = 2, 3, \dots, 10$ outcomes, compute the distance from the corner $\delta_1$ to the uniform distribution on $n$ outcomes with `fisher_rao_distance`, and compare it with $2\arccos(1/\sqrt n)$. What happens as $n$ grows, and why can the answer never exceed $\pi$? Then explain why the uniform distribution is the only point of the triangle equidistant from the three corners.
2. **Angle, chord and the triangle inequality.** Show on paper that $\mathrm{Hel}(p, q) = \sqrt2\,\sin(d_{FR}(p, q)/4)$. Then search random triples for the one closest to *breaking* the triangle inequality for $d_{FR}$: where does its middle point sit?
3. **A coordinate that does not change.** Take $p = (0.3, 0.6, 0.1)$ and $q = (0.6, 0.3, 0.1)$. Predict the geodesic midpoint by hand with the paper formula, then check it with `fisher_rao_geodesic`. Repeat with $p_3 = 0.3$ instead of $0.1$ (renormalise the other two entries): is the bulge factor $2/(1 + B)$ larger or smaller, and why, in terms of the overlap?
4. **Harder: the geodesic is a minimum, and it has constant speed.** (a) Show by hand (differentiate the two sines) that the slerp $\phi_t$ has unit norm for every $t$ and that its speed $\|d\phi_t/dt\|$ equals $\alpha$, so the Fisher speed of $p_t$ is the constant $2\alpha = d_{FR}$. (b) Perturb the geodesic of pair A: add $\varepsilon\sin(\pi t)\,w$ to $\phi_t$, for instance with $w = (0, 0, 1)$ (toward outcome $3$), renormalise to the sphere, square, and compute the Fisher length with `fisher_length` for $\varepsilon \in [-0.2, 0.2]$. Plot length against $\varepsilon$, check that the minimum is at $\varepsilon = 0$, and estimate how the excess length grows with $\varepsilon$ (linearly or quadratically?). (c) Explain, with the $(r, \omega)$ form of the metric, why a geodesic between two points of the same edge never leaves that edge.

## What you built

- You can derive $dp_i^2/p_i = 4\,d\phi_i^2$ and say where the factor $2$ comes from: a sphere of radius $2$.
- You can check the closed form on a coin, where the integral that `02/06` left open equals $2\arcsin\sqrt\theta$, and you confirmed it three ways to six decimals.
- You can give the distance from any distribution to a corner, $2\arccos\sqrt{p_i}$, explain why it is finite although the metric's coefficients blow up at the boundary, and why no two distributions are more than $\pi$ apart.
- You can read the Bhattacharyya coefficient as the cosine of half the distance and the Hellinger distance as the chord of the same sphere divided by $\sqrt2$, which gives the Hellinger triangle inequality and the $\tfrac18$ of `02/06` for free.
- You can compute a geodesic point by hand by spherical interpolation and squaring, write its midpoint on paper, and explain the bulge with the flight-route picture and the two costs of the $(r, \omega)$ form of the metric.
- You can say how big the bulge is: on pair A the geodesic lifts $p_3$ from $0.05$ to $0.0612$ and the straight path is $0.19\%$ longer; over random pairs the excess has a median of $0.16\%$ and reaches $4.23\%$.

You started with a metric and finished with a whole geometry, built by hand.

## What's next

Everything here compares distributions outcome by outcome: the sphere does not know whether outcome $1$ is next to outcome $2$ or far from it. Relabel the outcomes and every distance stays the same, and between two distributions with disjoint supports the geodesic fades one into the other, as the corner-to-corner midpoint $(0.5, 0.5, 0)$ showed. In `02/08_two_geometries_one_simplex` the outcomes acquire positions on a line, and a second geometry appears on the same simplex, one in which moving mass to a neighbouring outcome is cheaper than moving it far away. Comparing the two is the hinge into optimal transport.

## References

- C. R. Rao, *Information and accuracy attainable in the estimation of statistical parameters*, Bull. Calcutta Math. Soc. **37**, 81–91 (1945) — introduces the Fisher information as a Riemannian metric on distributions.
- A. Bhattacharyya, *On a measure of divergence between two statistical populations defined by their probability distributions*, Bull. Calcutta Math. Soc. **35**, 99–109 (1943) — introduces the coefficient $\sum_i\sqrt{p_i q_i}$.
- S.-i. Amari, *Information Geometry and Its Applications*, Applied Mathematical Sciences, Springer (2016) — the Fisher metric on the simplex and its invariance. doi:10.1007/978-4-431-55978-8
- K. Shoemake, *Animating rotation with quaternion curves*, ACM SIGGRAPH Computer Graphics **19**, 245–254 (1985) — the spherical linear interpolation (slerp). doi:10.1145/325165.325242

**Previous:** `notebooks/02_InformationAndGeometry/06_fisher_rao_metric.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/08_two_geometries_one_simplex.ipynb`